# Project Work

## Model 3: Advanced Feature Engineering + LightGBM / XGBoost Ensemble

This notebook is the custom experimental model. It combines lexical, semantic, and relative-ranking features, then blends tree-based rankers for final top-three predictions.


# 1. Setup & Configuration

## 1.1 Runtime Dependency Installation


In [ ]:
# ── Section 1: Setup & Configuration ────────────────────────────────────
# rank_bm25 and sentence-transformers are not in Kaggle's default image;
# installing at runtime keeps the kernel portable across environments.
# Install dependencies not available by default on Kaggle
import subprocess
subprocess.run(["pip", "install", "rank_bm25", "sentence-transformers", "-q"], check=True)
print("Dependencies ready")

## 1.2 Library Imports


In [ ]:
# Core scientific stack + LightGBM + XGBoost + W&B
import os
import re
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import wandb

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupKFold

print("All imports OK")

## 1.3 Kaggle and Local Path Resolution


In [ ]:
# then local project paths, so the same notebook runs on Kaggle and offline.
TRAIN_PATH      = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH       = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

for _var, _alts in [
    ("TRAIN_PATH",      ["data/train.csv",             "../../data/train.csv"]),
    ("TEST_PATH",       ["data/test.csv",              "../../data/test.csv"]),
    ("SAMPLE_SUB_PATH", ["data/sample_submission.csv", "../../data/sample_submission.csv"]),
]:
    if not os.path.exists(globals()[_var]):
        for _p in _alts:
            if os.path.exists(_p):
                globals()[_var] = _p
                break

print(f"TRAIN → {TRAIN_PATH}")
print(f"TEST  → {TEST_PATH}")


## 1.4 W&B Configuration and Hyperparameters


In [ ]:
# W&B key resolution: Kaggle Secrets -> os.environ -> disabled mode.
# WANDB_API_KEY is available from Kaggle Secrets or the local environment.
wandb_mode = "disabled"
try:
    from kaggle_secrets import UserSecretsClient
    env_key = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    env_key = os.environ.get("WANDB_API_KEY")

if env_key:
    wandb.login(key=env_key, relogin=True)
    wandb_mode = "online"
else:
    print("W&B key not detected. Running in disabled mode.")

wandb.init(
    project="DL-GenAI-Project",
    name="Model_3_MiniLM_RelativeRank_Ensemble",
    mode=wandb_mode,
    config={
        "word_tfidf_ngram":     "(1,2)",
        "word_tfidf_features":  10000,
        "char_tfidf_ngram":     "(2,4)",
        "char_tfidf_features":  6000,
        "sentence_model":       "all-MiniLM-L6-v2",
        "gkf_folds":            5,
        "lgb_n_estimators":     500,
        "lgb_lr":               0.05,
        "lgb_num_leaves":       63,
        "lgb_colsample":        0.7,
        "xgb_n_estimators":     400,
        "xgb_lr":               0.05,
        "xgb_max_depth":        6,
        "blend_steps":          17,
    }
)
CFG = wandb.config
print("W&B run:", wandb.run.name, "| mode:", wandb_mode)


# 2. Data Pipeline


## 2.1 CSV Loading and Schema Check


In [ ]:
# ── Section 2: Data Pipeline ───────────────────────────────────────────
# Load CSV data; inject empty 'context' column if absent for uniform field access.
# construction so the model never learns from accidental "nan" tokens.
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

OPTION_COLS = ["A", "B", "C", "D", "E"]
TEXT_COLS = ["context", "prompt", *OPTION_COLS]

for df in [train_df, test_df]:
    if "context" not in df.columns:
        df["context"] = ""
    for col in TEXT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("")

print("Train:", train_df.shape, "| Test:", test_df.shape)
train_df.head(3)


## 2.2 Text Normalisation and Lexical Features


Word TF-IDF captures terms and short phrases, character TF-IDF helps with spelling variants and abbreviations, and BM25 limits the effect of repeated terms.


In [ ]:
# prevent out-of-vocabulary token drops on unseen test options.
# char_wb analyzer pads word boundaries to avoid cross-word character noise.
def clean(text: str) -> str:
    """Lowercase + collapse whitespace. Retains punctuation for char n-grams."""
    return re.sub(r"\s+", " ", str(text).lower().strip())

# Build combined query (context + prompt) for each question
train_queries = (train_df["context"] + " " + train_df["prompt"]).map(clean).tolist()
test_queries  = (test_df["context"]  + " " + test_df["prompt"]).map(clean).tolist()

# Gather all text to fit a shared vocabulary — prevents OOV on test options
all_opt_texts = []
for df in [train_df, test_df]:
    for col in OPTION_COLS:
        all_opt_texts += df[col].fillna("").map(clean).tolist()

full_corpus = train_queries + test_queries + all_opt_texts

# Word TF-IDF: 1–2 ngrams capture individual keywords and two-word phrases
word_tfidf = TfidfVectorizer(
    max_features=CFG.word_tfidf_features,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True,
    strip_accents="unicode",
)
word_tfidf.fit(full_corpus)

# Char TF-IDF: 2–4 character n-grams handle morphological variants and
# technical abbreviations that word tokenisation splits poorly
char_tfidf = TfidfVectorizer(
    max_features=CFG.char_tfidf_features,
    ngram_range=(2, 4),
    analyzer="char_wb",   # char_wb pads word boundaries to reduce cross-word noise
    sublinear_tf=True,
)
char_tfidf.fit(full_corpus)

print(f"Word vocab: {len(word_tfidf.vocabulary_)}  |  Char vocab: {len(char_tfidf.vocabulary_)}")

## 2.3 Semantic Feature Engineering


MiniLM adds a semantic similarity score between each question and option.


In [ ]:
# Load all-MiniLM-L6-v2 — 384-dimensional sentence embeddings, fast on CPU
minilm = SentenceTransformer("all-MiniLM-L6-v2")
print("MiniLM loaded")

# Encode all queries and all option texts in bulk — batch encoding is ~10x
# faster than encoding one string at a time inside a Python loop
all_train_opts = []
for _, row in train_df.iterrows():
    for col in OPTION_COLS:
        all_train_opts.append(clean(str(row.get(col, ""))))

all_test_opts = []
for _, row in test_df.iterrows():
    for col in OPTION_COLS:
        all_test_opts.append(clean(str(row.get(col, ""))))

print(f"Encoding {len(train_queries)} train queries...")
train_q_embs = minilm.encode(train_queries,   batch_size=64, show_progress_bar=True,
                              normalize_embeddings=True)
print(f"Encoding {len(test_queries)} test queries...")
test_q_embs  = minilm.encode(test_queries,    batch_size=64, show_progress_bar=True,
                              normalize_embeddings=True)
print(f"Encoding {len(all_train_opts)} train option texts...")
train_o_embs = minilm.encode(all_train_opts,  batch_size=64, show_progress_bar=True,
                              normalize_embeddings=True)
print(f"Encoding {len(all_test_opts)} test option texts...")
test_o_embs  = minilm.encode(all_test_opts,   batch_size=64, show_progress_bar=True,
                              normalize_embeddings=True)

# For normalized embeddings: cosine similarity == dot product
# Shape: train_q_embs[i] ∙ train_o_embs[i*5 + j] = similarity of question i with option j
def minilm_cos(q_embs, o_embs_flat, n_questions):
    """Return (n_questions * 5,) float32 array of MiniLM cosine similarities."""
    sims = []
    for i in range(n_questions):
        for j in range(5):                            # 5 options per question
            sim = float(np.dot(q_embs[i], o_embs_flat[i * 5 + j]))
            sims.append(sim)
    return np.array(sims, dtype=np.float32)

train_minilm_sims = minilm_cos(train_q_embs, train_o_embs, len(train_df))
test_minilm_sims  = minilm_cos(test_q_embs,  test_o_embs,  len(test_df))

print(f"MiniLM sims — train mean: {train_minilm_sims.mean():.4f}  std: {train_minilm_sims.std():.4f}")


# 3. Model Architecture & Definition

## 3.1 Feature Extraction and Long-Format Construction


In [ ]:
# ── Section 3: Feature Extraction — Combine All Feature Sets ────────────
# Base features: lexical (TF-IDF cosine, BM25), semantic (MiniLM cosine),
# and structural (option/prompt length and word-count ratios).
def jaccard(set_a: set, set_b: set) -> float:
    """Jaccard = |A ∩ B| / |A ∪ B|. High score means option reuses prompt vocabulary."""
    if not set_a and not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def extract_features(
    df: pd.DataFrame,
    queries: list,
    minilm_sims_flat: np.ndarray,
    is_test: bool = False,
) -> pd.DataFrame:
    """
    Long-format: one row per (question, option) pair.

    Base features:
      word_cos     — word TF-IDF cosine sim
      char_cos     — char TF-IDF cosine sim
      bm25         — raw BM25 score
      bm25_norm    — BM25 normalised within question (0-1)
      jaccard      — word set Jaccard index
      word_intersect — fraction of option words in query
      minilm_cos   — MiniLM 384-d embedding cosine sim (single scalar)
      opt_len / prompt_len / len_ratio
      opt_wc  / prompt_wc  / wc_ratio
    """
    word_q_vecs = word_tfidf.transform(queries)
    char_q_vecs = char_tfidf.transform(queries)

    records = []
    for i, (_, row) in enumerate(df.iterrows()):
        prompt_txt   = clean(str(row.get("prompt", "")))
        query_tokens = prompt_txt.split()
        query_words  = set(query_tokens)
        p_len, p_wc  = len(prompt_txt), len(query_tokens)

        wq_vec = word_q_vecs[i]
        cq_vec = char_q_vecs[i]

        opt_texts = [clean(str(row.get(opt, ""))) for opt in OPTION_COLS]

        # BM25: scores all 5 options at once — O(5*vocab) per question
        bm25_raw  = BM25Okapi([t.split() for t in opt_texts]).get_scores(query_tokens)
        bm25_max  = bm25_raw.max() + 1e-9
        bm25_norm = bm25_raw / bm25_max

        for j, opt in enumerate(OPTION_COLS):
            opt_txt   = opt_texts[j]
            opt_words = set(opt_txt.split())
            o_len     = len(opt_txt)
            o_wc      = len(opt_txt.split())

            wo_vec   = word_tfidf.transform([opt_txt])
            co_vec   = char_tfidf.transform([opt_txt])

            label = 0
            if not is_test and "answer" in row:
                label = 1 if str(row["answer"]).strip().upper() == opt else 0

            records.append({
                "id":             row["id"],
                "option_key":     opt,
                # Feature Set A — Lexical
                "word_cos":       float(cosine_similarity(wq_vec, wo_vec)[0][0]),
                "char_cos":       float(cosine_similarity(cq_vec, co_vec)[0][0]),
                "bm25":           float(bm25_raw[j]),
                "bm25_norm":      float(bm25_norm[j]),
                "jaccard":        jaccard(query_words, opt_words),
                "word_intersect": len(query_words & opt_words) / max(len(opt_words), 1),
                # Feature Set B — Semantic
                "minilm_cos":     float(minilm_sims_flat[i * 5 + j]),
                # Structural
                "opt_len":        o_len,
                "prompt_len":     p_len,
                "len_ratio":      o_len / (p_len + 1),
                "opt_wc":         o_wc,
                "prompt_wc":      p_wc,
                "wc_ratio":       o_wc / (p_wc + 1),
                "label":          label,
            })

    return pd.DataFrame(records)

print("Extracting train features...")
train_feat = extract_features(train_df, train_queries, train_minilm_sims, is_test=False)
print("Extracting test  features...")
test_feat  = extract_features(test_df,  test_queries,  test_minilm_sims,  is_test=True)

print(f"Train: {train_feat.shape}  positives={train_feat['label'].sum()}")
print(f"Test : {test_feat.shape}")
train_feat.head(10)

## 3.2 Relative Ranking Feature Augmentation


Relative ranks and differences from the question mean show which option is strongest among its five alternatives.


In [ ]:
# reveal *which option is best within a question* regardless of absolute scale.
# bm25_rank=1 means this option scored highest among all 5 for that question.
# Features we will rank and compute diff-from-mean for
RANK_FEATS = ["word_cos", "char_cos", "bm25", "bm25_norm",
              "jaccard", "word_intersect", "minilm_cos"]

def add_relative_features(df: pd.DataFrame, feats: list) -> pd.DataFrame:
    """
    For each feature in `feats`, add two new columns grouped by question id:
      {feat}_diff  = value - per-question mean   (positive = above average)
      {feat}_rank  = descending rank within the question group (1 = best option)

    Viva: A bm25_rank of 1 means this option has the highest BM25 score among
    all five options for that question, regardless of its absolute numeric value.
    This makes the feature invariant to question-level scale differences.
    """
    grp = df.groupby("id")
    for feat in feats:
        df[f"{feat}_diff"] = df[feat] - grp[feat].transform("mean")
        df[f"{feat}_rank"] = grp[feat].rank(ascending=False, method="min")
    return df

train_feat = add_relative_features(train_feat, RANK_FEATS)
test_feat  = add_relative_features(test_feat,  RANK_FEATS)

# Full feature column list: base + relative
BASE_COLS = ["word_cos", "char_cos", "bm25", "bm25_norm", "jaccard",
             "word_intersect", "minilm_cos",
             "opt_len", "prompt_len", "len_ratio",
             "opt_wc",  "prompt_wc",  "wc_ratio"]
RANK_COLS = [f"{f}{s}" for f in RANK_FEATS for s in ("_diff", "_rank")]
FEAT_COLS = BASE_COLS + RANK_COLS

print(f"Total features: {len(FEAT_COLS)}")
print(FEAT_COLS)

# 4. Training and Evaluation


## 4.1 GroupKFold Cross-Validation Training Loop


GroupKFold keeps all five options from a question in the same fold. Class weighting and early stopping are applied within each fold.


In [ ]:
# ── Section 4: Training & Validation Loop ────────────────────────────────
# preventing leakage between train and validation rows.
X_all  = train_feat[FEAT_COLS].values.astype(np.float32)
y_all  = train_feat["label"].values
X_test = test_feat[FEAT_COLS].values.astype(np.float32)
groups = train_feat["id"].values   # group identifier = question id

print(f"Feature matrix — train: {X_all.shape}  test: {X_test.shape}")
print(f"Label balance  — 0: {(y_all==0).sum()}  1: {(y_all==1).sum()}")

N_FOLDS = CFG.gkf_folds
gkf     = GroupKFold(n_splits=N_FOLDS)

oof_lgb = np.zeros(len(X_all), dtype=np.float32)
oof_xgb = np.zeros(len(X_all), dtype=np.float32)
test_lgb_preds = np.zeros(len(X_test), dtype=np.float32)
test_xgb_preds = np.zeros(len(X_test), dtype=np.float32)

spw_all = (y_all == 0).sum() / max((y_all == 1).sum(), 1)  # global scale_pos_weight

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_all, y_all, groups=groups)):
    X_tr, X_val = X_all[tr_idx], X_all[val_idx]
    y_tr, y_val = y_all[tr_idx], y_all[val_idx]
    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

    # ── LightGBM fold ──────────────────────────────────────────────────────
    lgb_clf = lgb.LGBMClassifier(
        objective        = "binary",
        metric           = "binary_logloss",
        n_estimators     = CFG.lgb_n_estimators,
        learning_rate    = CFG.lgb_lr,
        num_leaves       = CFG.lgb_num_leaves,
        colsample_bytree = CFG.lgb_colsample,
        subsample        = 0.8,
        class_weight     = "balanced",
        random_state     = 42 + fold,
        verbose          = -1,
    )
    lgb_clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(200)],
    )
    oof_lgb[val_idx]    = lgb_clf.predict_proba(X_val)[:, 1]
    test_lgb_preds     += lgb_clf.predict_proba(X_test)[:, 1] / N_FOLDS

    # ── XGBoost fold ───────────────────────────────────────────────────────
    xgb_clf = xgb.XGBClassifier(
        objective             = "binary:logistic",
        eval_metric           = "logloss",
        n_estimators          = CFG.xgb_n_estimators,
        learning_rate         = CFG.xgb_lr,
        max_depth             = CFG.xgb_max_depth,
        subsample             = 0.8,
        colsample_bytree      = 0.7,
        scale_pos_weight      = spw,
        use_label_encoder     = False,
        random_state          = 42 + fold,
        early_stopping_rounds = 30,
        verbosity             = 0,
    )
    xgb_clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    oof_xgb[val_idx]    = xgb_clf.predict_proba(X_val)[:, 1]
    test_xgb_preds     += xgb_clf.predict_proba(X_test)[:, 1] / N_FOLDS

    fold_acc_lgb = accuracy_score(y_val, (oof_lgb[val_idx] >= 0.5).astype(int))
    fold_acc_xgb = accuracy_score(y_val, (oof_xgb[val_idx] >= 0.5).astype(int))
    print(f"Fold {fold+1}/{N_FOLDS}  LGB acc: {fold_acc_lgb:.4f}  XGB acc: {fold_acc_xgb:.4f}")

print("\nGroupKFold OOF complete")


## 4.2 Blend Weight Search and OOF Metric Evaluation


The blend weight is selected on out-of-fold MAP@3, then used to rank the test options.


In [ ]:
# so the final mix is validation-driven rather than hand-picked.
# the top-three ranked labels, matching the Kaggle scoring rule.
def ap_at_3(group: pd.DataFrame) -> float:
    """Average Precision@3: rewards placing the correct option earlier in the top-3."""
    ranked = group.sort_values("pred_proba", ascending=False).reset_index(drop=True)
    score, hits = 0.0, 0
    for rank, row in ranked.head(3).iterrows():
        if row["label"] == 1:
            hits += 1
            score += hits / (rank + 1)
    return score

# Sweep alpha (LGB weight) from 0.05 → 0.95 to find the best blend on OOF data
best_alpha, best_map3 = 0.5, -1.0
oof_df = train_feat[["id", "option_key", "label"]].copy()

for alpha in np.arange(0.05, 1.0, 0.05):
    oof_df["pred_proba"] = alpha * oof_lgb + (1 - alpha) * oof_xgb
    map3 = oof_df.groupby("id").apply(ap_at_3).mean()
    if map3 > best_map3:
        best_map3  = map3
        best_alpha = round(float(alpha), 2)

# Final OOF blend using optimal alpha
oof_df["pred_proba"] = best_alpha * oof_lgb + (1 - best_alpha) * oof_xgb
oof_preds = (oof_df["pred_proba"] >= 0.5).astype(int)

oof_acc = accuracy_score(y_all, oof_preds)
oof_f1  = f1_score(y_all, oof_preds, average="macro", zero_division=0)
oof_ll  = log_loss(y_all, oof_df["pred_proba"])

print(f"Best alpha (LGB weight) : {best_alpha}")
print(f"OOF MAP@3               : {best_map3:.4f}")
print(f"OOF accuracy            : {oof_acc:.4f}")
print(f"OOF macro-F1            : {oof_f1:.4f}")
print(f"OOF logloss             : {oof_ll:.4f}")

wandb.log({
    "oof_map_at_3":   best_map3,
    "oof_accuracy":   oof_acc,
    "oof_f1_macro":   oof_f1,
    "oof_logloss":    oof_ll,
    "best_lgb_alpha": best_alpha,
})


# 5. Conclusion


## 5.1 Test Inference, Blend, and Submission Export


Fold predictions are averaged, blended with the selected alpha, and ranked to produce the top three labels.


In [ ]:
# ── Section 5: Inference & Submission ───────────────────────────────────
# Apply the optimal alpha blend to averaged test-fold predictions.
# Top-3 options per question are selected by descending pred_proba and
# joined as a space-separated string matching the competition format.
# the exact Kaggle-required schema: ID and Prediction.
test_feat["pred_proba"] = best_alpha * test_lgb_preds + (1 - best_alpha) * test_xgb_preds

def top3(group: pd.DataFrame) -> str:
    return " ".join(
        group.sort_values("pred_proba", ascending=False)["option_key"].head(3).tolist()
    )

submission = (
    test_feat
    .groupby("id", sort=False)
    .apply(top3)
    .reset_index()
)

if os.path.exists(SAMPLE_SUB_PATH):
    sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
    id_col, pred_col = sample_sub.columns[0], sample_sub.columns[1]
else:
    id_col, pred_col = "ID", "Prediction"

submission.columns = [id_col, pred_col]
assert len(submission) == len(test_df), f"Row count mismatch: expected {len(test_df)}, got {len(submission)}"
submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv — {len(submission)} rows")
print(submission.head(10))

wandb.finish()
